# Install Ollama Server to host model (LLava)

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:13 https://developer.download.nvidia.com/compute/cuda/repos/ubu

In [ ]:
!pip install fastapi uvicorn httpx python-multipart pyngrok nest-asyncio websockets -q

In [ ]:
!pip install fastapi uvicorn httpx python-multipart pyngrok nest-asyncio opencv-python-headless numpy aiofiles -q
print("Dependencies installed")

Dependencies installed


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Server.py Code

In [ ]:
%%writefile /content/server.py
"""
FastAPI middleware server — receives image from the Raspberry Pi,
queries an Ollama vision model, stores recent alerts in memory,
and serves them to a frontend dashboard.
"""

from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
#from fastapi.responses import JSONResponse
#from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse, Response
from fastapi.middleware.cors import CORSMiddleware

import httpx
import base64
import logging
import traceback
import tempfile
import os
import uuid
from datetime import datetime
from collections import deque
from pathlib import Path

logging.basicConfig(level=logging.DEBUG)
log = logging.getLogger("server")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL        = "llava:7b"
API_KEY      = "change-me-to-something-secret"

VIDEO_KEY_FRAMES   = 4
SNAPSHOTS_DIR      = Path("/content/snapshots")
MAX_ALERT_HISTORY  = 100

SNAPSHOTS_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_IMAGE = (
    "You are a security camera AI. Describe what is happening in this image "
    "in 1-2 sentences. Focus on: people, animals, vehicles, or unusual activity."
)

PROMPT_VIDEO = (
    "You are a security camera AI. I'm showing you {n} frames sampled evenly "
    "from a short motion-triggered video clip. Describe what is happening "
    "across these frames in 2-3 sentences. Focus on movement, people, animals, "
    "vehicles, or any unusual activity. Note any changes between frames."
)

# In-memory alert store, newest appended last
alert_history = deque(maxlen=MAX_ALERT_HISTORY)


# ---------------------------------------------------------------------------
# FastAPI app
# ---------------------------------------------------------------------------

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ---------------------------------------------------------------------------
# Video frame extraction (kept for future use; not called by the image-only Pi)
# ---------------------------------------------------------------------------

def extract_key_frames(video_bytes, n_frames=VIDEO_KEY_FRAMES):
    import cv2

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
            tmp.write(video_bytes)
            tmp_path = tmp.name

        cap = cv2.VideoCapture(tmp_path)
        if not cap.isOpened():
            new_path = tmp_path.replace(".mp4", ".avi")
            os.rename(tmp_path, new_path)
            tmp_path = new_path
            cap = cv2.VideoCapture(tmp_path)
            if not cap.isOpened():
                return []

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            return []

        n_frames = min(n_frames, total_frames)
        sample_idxs = [int(total_frames * (i + 0.5) / n_frames) for i in range(n_frames)]

        b64_frames = []
        for idx in sample_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, frame = cap.read()
            if not ok:
                continue
            _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            b64_frames.append(base64.b64encode(buf.tobytes()).decode())

        cap.release()
        return b64_frames

    except Exception:
        log.error("Frame extraction failed: %s", traceback.format_exc())
        return []
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)


# ---------------------------------------------------------------------------
# Ollama
# ---------------------------------------------------------------------------

async def query_ollama(b64_images, prompt):
    log.debug("Sending %d image(s) to Ollama '%s'", len(b64_images), MODEL)
    try:
        async with httpx.AsyncClient(timeout=180) as client:
            r = await client.post(OLLAMA_URL, json={
                "model":  MODEL,
                "prompt": prompt,
                "images": b64_images,
                "stream": False,
            })
            r.raise_for_status()
            description = r.json()["response"].strip()
            log.info("Ollama description: %s", description)
            return description

    except httpx.ConnectError:
        raise HTTPException(status_code=503, detail="Ollama is not running")
    except httpx.TimeoutException:
        raise HTTPException(status_code=504, detail="Ollama timed out")
    except httpx.HTTPStatusError as exc:
        body = exc.response.text[:400]
        raise HTTPException(status_code=502, detail=f"Ollama returned {exc.response.status_code}: {body}")
    except Exception as e:
        log.error("Ollama query failed: %s", traceback.format_exc())
        raise HTTPException(status_code=500, detail=f"Unexpected error: {e}")


# ---------------------------------------------------------------------------
# Routes
# ---------------------------------------------------------------------------

@app.get("/health")
async def health():
    try:
        async with httpx.AsyncClient(timeout=10) as client:
            r = await client.get("http://localhost:11434/api/tags")
            models = [m["name"] for m in r.json().get("models", [])]
        return {
            "fastapi":     "ok",
            "ollama":      "ok",
            "models":      models,
            "alert_count": len(alert_history),
        }
    except Exception as e:
        return {"fastapi": "ok", "ollama": "error", "detail": str(e)}


@app.get("/alerts")
async def get_alerts(limit: int = 50):
    """Return recent alerts, newest first."""
    items = list(alert_history)[-limit:]
    items.reverse()
    return {"alerts": items}


@app.delete("/alerts")
async def clear_alerts():
    """Clear all stored alerts (snapshots on disk are kept)."""
    n = len(alert_history)
    alert_history.clear()
    return {"cleared": n}


@app.post("/analyze")
async def analyze(
    image:          UploadFile = File(...),
    timestamp:      str        = Form(...),
    motion_regions: int        = Form(...),
    x_api_key:      str        = Header(...),
    video:          UploadFile = File(None),
):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")

    # --- Read snapshot ---
    try:
        image_bytes = await image.read()
        image_b64   = base64.b64encode(image_bytes).decode()
        log.debug("Snapshot received: %d bytes", len(image_bytes))
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read image field")

    # --- Save snapshot to disk so frontend can render it ---
    alert_id  = uuid.uuid4().hex[:12]
    base_name = datetime.utcnow().strftime("%Y%m%d_%H%M%S") + f"_{alert_id}"
    snap_path = SNAPSHOTS_DIR / f"{base_name}.jpg"
    with open(snap_path, "wb") as f:
        f.write(image_bytes)
    image_url = f"/snapshots/{snap_path.name}"

    # --- Decide image vs video pipeline ---
    use_video  = False
    b64_images = [image_b64]
    prompt     = PROMPT_IMAGE

    if video is not None:
        try:
            video_bytes = await video.read()
            if len(video_bytes) > 0:
                key_frames = extract_key_frames(video_bytes, VIDEO_KEY_FRAMES)
                if key_frames:
                    use_video  = True
                    b64_images = key_frames
                    prompt     = PROMPT_VIDEO.format(n=len(key_frames))
        except Exception:
            log.error("Video processing failed: %s", traceback.format_exc())

    # --- Query Ollama ---
    description = await query_ollama(b64_images, prompt)

    # --- Build alert record + store + return ---
    alert = {
        "id":             alert_id,
        "timestamp":      timestamp,
        "description":    description,
        "motion_regions": motion_regions,
        "image_url":      image_url,
        "used_video":     use_video,
        "frames_used":    len(b64_images),
        "model":          MODEL,
    }

    alert_history.append(alert)
    log.info("Alert stored [%s] %d regions — %s", alert_id, motion_regions, description[:80])

    return JSONResponse({
        "status":      "ok",
        "alert":       alert,
        "description": description,
        "used_video":  use_video,
        "frames_used": len(b64_images),
    })

@app.get("/snapshots/{filename}")
async def serve_snapshot(filename: str):
    img_path = SNAPSHOTS_DIR / filename
    if not img_path.exists():
        raise HTTPException(status_code=404, detail="Image not found")
    return Response(content=img_path.read_bytes(), media_type="image/jpeg")


Overwriting /content/server.py


# Pull the llava7b model using Ollama

In [ ]:
import subprocess, time

# Start Ollama server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)  # wait for it to be ready

# Pull the model (this takes a few minutes the first time)
!ollama pull llava:7b   # use llava:7b on free Colab — 13b may OOM on T4

# Ngrok Setup to open the ports for the app communication

In [ ]:
import nest_asyncio, uvicorn, threading, os
from pyngrok import ngrok

nest_asyncio.apply()


# Set your ngrok auth token (free at https://dashboard.ngrok.com)
ngrok.set_auth_token("3D2iiUpB2U8ex4qjk9y7VeNIj8I_4am9ZL3S8wtBuk2JwsuGK")

# Kill any existing ngrok tunnels to avoid endpoint limits
ngrok.kill()

# Open tunnel on port 8000
public_url = ngrok.connect(8000)
print(f"\n=== Your Pi's WEBHOOK_URL ===")
print(f"{public_url}/analyze\n")

# Run FastAPI in a background thread
def run():
    uvicorn.run("server:app", host="0.0.0.0", port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()


=== Your Pi's WEBHOOK_URL ===
NgrokTunnel: "https://september-festivity-legislate.ngrok-free.dev" -> "http://localhost:8000"/analyze



In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Ollama health check (sanity check) making sure the correct model is running on ollama

In [ ]:
import httpx, asyncio, base64

async def test_ollama():
    async with httpx.AsyncClient(timeout=60) as client:
        # First check Ollama is alive
        r = await client.get("http://localhost:11434/api/tags")
        print("Models available:", r.json())

asyncio.run(test_ollama())

Models available: {'models': [{'name': 'llava:7b', 'model': 'llava:7b', 'modified_at': '2026-04-29T20:01:01.394469255Z', 'size': 4733363377, 'digest': '8dd30f6b0cb19f555f2c7a7ebda861449ea2cc76bf1f44e262931f45fc81d081', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama', 'clip'], 'parameter_size': '7B', 'quantization_level': 'Q4_0'}}]}


# Run the server code

In [ ]:
import subprocess, time, threading

# Kill any zombie uvicorn process from a previous run
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(1)

proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "debug"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="/content",
    bufsize=1,
)

def stream_logs():
    for line in proc.stdout:
        print(line, end="")

threading.Thread(target=stream_logs, daemon=True).start()
time.sleep(6)

if proc.poll() is not None:
    print(f"\n❌ Server died with exit code {proc.returncode} — see error above")
else:
    print("\n✅ Server is up on port 8000")

INFO:     Started server process [40810]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     130.68.183.207:0 - "OPTIONS /alerts?limit=50 HTTP/1.1" 200 OK
INFO:     130.68.183.207:0 - "GET /alerts?limit=50 HTTP/1.1" 200 OK

✅ Server is up on port 8000


In [ ]:
with open('/content/index.html', 'r') as f:
    print(f.read())

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Motion Detection Feed</title>
  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin />
  <link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500&family=Inter:wght@400;500;600&display=swap" rel="stylesheet" />
  <style>
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

    :root {
      --bg:        #0c0d10;
      --surface:   #14161a;
      --surface2:  #1c1f24;
      --border:    rgba(255,255,255,0.08);
      --border2:   rgba(255,255,255,0.16);
      --text:      #e8e9eb;
      --muted:     #8b8e94;
      --hint:      #555861;
      --accent:    #4a9eff;
      --success:   #4ade80;
      --warning:   #fbbf24;
      --danger:    #f87171;
      --mono:      'JetBrains Mono', monospace;
      --sans:      '

# Testing server API

In [ ]:
import requests, httpx, base64, asyncio

async def test():
    async with httpx.AsyncClient(timeout=60) as client:
        img = await client.get("https://fastly.picsum.photos/id/987/640/480.jpg?hmac=UBObHCH4yXZSk45rkCY688eM_MuZkgQuVqdfjfzxz-w")
        b64 = base64.b64encode(img.content).decode()

    r = requests.post("http://localhost:11434/api/generate", json={
        "model": "llava:7b",   # <-- exact name from Cell 1
        "prompt": "Describe this image in one sentence.",
        "images": [b64],
        "stream": False
    })
    print("Status:", r.status_code)
    print("Response:", r.text[:500])

asyncio.run(test())

Status: 200
Response: {"model":"llava:7b","created_at":"2026-04-29T21:51:12.670039805Z","response":" A beautiful view of sunset over the top of a mountain, with clouds and hues of pink and orange in the sky. ","done":true,"done_reason":"stop","context":[733,16289,28793,733,5422,28733,28734,28793,22836,456,3469,297,624,12271,28723,733,28748,16289,28793,330,4672,2204,302,4376,673,754,272,1830,302,264,10660,28725,395,16393,304,295,1139,302,12937,304,14545,297,272,7212,28723,28705],"total_duration":1297351528,"load_durat


In [ ]:
import subprocess, time
result = subprocess.run(["pgrep", "-f", "ollama serve"], capture_output=True, text=True)
if result.stdout.strip():
    print(f"✅ Ollama running (PID {result.stdout.strip()})")
else:
    print("❌ Ollama not running — restarting...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    print("✅ Ollama restarted")

✅ Ollama running (PID 8007)


In [ ]:
!pkill -f uvicorn
import time; time.sleep(2)
# then your existing start-uvicorn cell

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [36843]
